# Bark Detection
Quick notebook that uses trained UrbanSound8K model to detect content various audio samples, intended to be used for Barking detection (but can be used for any of the audio classifications within that dataset).

### Training
Training is done in the console, not a notebook. 
A `click` cli has been written to perform basic tasks
1. Install Dependencies via `uv`:
  * `uv venv --python 3.13`
  * `source .venv/bin/activate`
  * `uv sync --all-extras`
2. Run training. model will be saved to `models/` directory
  * ex: `uv run -- barking train --num-epochs 20 --learn-rate 0.002`
3. Run inference.
  * Can be done via the cli: `uv run -- barking infer --audio-file data/bark.wav --sample-rate 22050`
  * Or can be done in this notebook.
    

In [1]:
import os
import torch
import librosa
from barking.dataset import UrbanSound8KDataset
from barking.ann import AudioClassifierANN

### Dataset. 
Download the UrbanSound8K dataset. Google it, etc. and extract into a directory `data` in the root of the repo

In [2]:
## some constants.
audio_dir = "../data/UrbanSound8K/audio"
metadata = "../data/UrbanSound8K/metadata/UrbanSound8K.csv"
labels = {
    0: "air_conditioner",
    1: "car_horn",
    2: "children_playing",
    3: "dog_bark",
    4: "drilling",
    5: "engine_idling",
    6: "gun_shot",
    7: "jackhammer",
    8: "siren",
    9: "street_music",
}
dataset = UrbanSound8KDataset(audio_dir=audio_dir, annotations=metadata)

### Model
loads the trained model from the checkpoint if it's available.

In [3]:
def load_model(model_path, input_size, num_classes, device):
    """Load the trained model from the saved checkpoint."""
    model = AudioClassifierANN(input_size=input_size, num_classes=num_classes)
    if os.path.exists(model_path) and os.path.isfile(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model

### Prediction func.
simply tries to retrieve the classification label and confidence score of a given audio sample

In [4]:
def predict(model_file, audio_file, sample_rate):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(device)
    waveform, sr = librosa.load(audio_file, sr=sample_rate)

    features = dataset.extract_features(waveform)
    features_tensor = torch.tensor(features, dtype=torch.float32).unsqueeze(0)
    features_tensor = features_tensor.to(device)

    model = load_model(model_file, input_size=202, num_classes=10, device=device)

    with torch.no_grad():
        outputs = model(features_tensor)

    _, predicted_class = torch.max(outputs, 1)
    confidence = torch.softmax(outputs, dim=1)[0][predicted_class].item()

    id = ([predicted_class.item()])[0]
    label_name = labels[int(id)]

    return (label_name, confidence)


guess, score = predict("../models/best_model.pth", "../data/bark.wav", sample_rate=22050)
print(guess, score)

cuda:0
dog_bark 0.994143545627594


#### Random
Randomly selects a sample from the training data and runs inference on it for fun

In [5]:
from random import randint

dataset = UrbanSound8KDataset(audio_dir=audio_dir, annotations=metadata)

randidx = randint(0, len(dataset))
# get a sample from the dataset
sample = dataset.metadata.iloc[randidx]
target = sample["class"]

guess, score = predict(
    "../models/best_model.pth",
    f"../data/UrbanSound8K/audio/fold{sample['fold']}/{sample['slice_file_name']}",
    sample_rate=22050,
)
print(f"Guess: {guess}; Actual: {target}; Confidence {score}")

cuda:0
Guess: dog_bark; Actual: dog_bark; Confidence 0.9950214624404907
